## 生成器和判别器模块

In [1]:
import torch
from torch import nn

这个代码块定义了一个生成器（`Generator`）类，它是用于生成数据（通常是图像）的神经网络架构，通常用于生成对抗网络（GAN）中。生成器的作用是从一个随机噪声向量中生成类似真实数据的样本。让我们逐步解释这个代码。

### 1. `class Generator(nn.Module)`
这个类继承自 PyTorch 的 `nn.Module`，表示这是一个神经网络模型。`Generator` 类实现了一个全连接的前馈神经网络。

### 2. `__init__(self)`
这是 `Generator` 类的构造函数，用来定义网络的结构和层。

- `super(Generator, self).__init__()`：调用父类 `nn.Module` 的构造函数，初始化模块。

#### `block(in_feat, out_feat, normalize=True)`
这是一个内部函数，用来创建每一层的基本块。这个块包括一个线性层（`nn.Linear`），一个可选的批量归一化层（`nn.BatchNorm1d`），以及一个 LeakyReLU 激活函数。每个块的具体作用：
- `nn.Linear(in_feat, out_feat)`：全连接层，将输入维度 `in_feat` 映射到输出维度 `out_feat`。
- `nn.BatchNorm1d(out_feat, 0.8)`：批量归一化层，标准化每批输入数据的均值和方差，增强模型的稳定性和收敛性。`0.8` 是动量参数。
- `nn.LeakyReLU(0.2, inplace=True)`：Leaky ReLU 激活函数，带有一个负斜率（`0.2`）。相比 ReLU，Leaky ReLU 允许负数部分具有小的梯度，避免“死神经元”问题。

#### `self.model = nn.Sequential(...)`
这里定义了整个生成器模型的层次结构，使用了 `nn.Sequential` 将多个层顺序堆叠起来：
- `*block(opt.latent_dim, 128, normalize=False)`：输入维度是 `latent_dim`，即噪声向量的维度（通常是低维的），输出维度为 128。这里不使用批量归一化。
- `*block(128, 256)`、`*block(256, 512)`、`*block(512, 1024)`：这些是逐步增大的全连接层，每个块会将输入数据进一步映射到更高的维度，并且使用批量归一化和 LeakyReLU 激活。
- `nn.Linear(1024, int(np.prod(img_shape)))`：最后一层，将维度为 1024 的输出映射到图像的扁平化维度，即 `np.prod(img_shape)`，这是图像的总像素数（宽度 × 高度 × 通道数）。
- `nn.Tanh()`：使用 Tanh 激活函数将输出限制在 [-1, 1] 范围内，通常生成的图像像素值会被缩放到这个范围。

### 3. `def forward(self, z)`
这是前向传播函数，定义了数据流经过网络的方式：
- `z` 是输入的噪声向量。
- `img = self.model(z)`：噪声向量 `z` 经过生成器模型 `self.model`，得到一个扁平化的图像向量。
- `img = img.view(img.size(0), *img_shape)`：将扁平的图像向量 `img` 重塑成 `img_shape`（通常是 `(channels, height, width)` 的格式），这样可以转换为适合显示或处理的图像。
- `return img`：返回生成的图像。

### 总结
这个生成器的作用是从随机噪声向量（`latent_dim`）开始，经过一系列全连接层，逐渐映射到一个与真实数据（如图像）具有相同维度的输出。通过 LeakyReLU 激活和 Tanh 的最后限制，生成器能够产生逼真的样本，通常用于 GAN 中与判别器（discriminator）对抗训练。



In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()

        def block(in_feat, out_feat, normalize=True):
            layers = [nn.Linear(in_feat, out_feat)]
            if normalize:
                layers.append(nn.BatchNorm1d(out_feat, 0.8))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(opt.latent_dim, 128, normalize=False),
            *block(128, 256),
            *block(256, 512),
            *block(512, 1024),
            nn.Linear(1024, int(np.prod(img_shape))),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), *img_shape)
        return img





这个代码块定义了一个判别器（`Discriminator`）类，它是用于判别数据（通常是图像）是否为真实数据的神经网络架构，通常用于生成对抗网络（GAN）中。判别器的作用是接收输入数据并输出一个概率，表示数据是否为真实的。

让我们逐步解释这个代码。

### 1. `class Discriminator(nn.Module)`
这个类继承自 PyTorch 的 `nn.Module`，表示这是一个神经网络模型。`Discriminator` 类实现了一个全连接的前馈神经网络，目的是判断输入数据的真实性。

### 2. `__init__(self)`
这是 `Discriminator` 类的构造函数，用来定义网络的结构和层。

- `super(Discriminator, self).__init__()`：调用父类 `nn.Module` 的构造函数，初始化模块。

#### `self.model = nn.Sequential(...)`
这里定义了整个判别器模型的层次结构，使用了 `nn.Sequential` 将多个层顺序堆叠起来：
- `nn.Linear(int(np.prod(img_shape)), 512)`：全连接层，将输入的图像数据（展平后的向量）从 `int(np.prod(img_shape))` 映射到 512 维。`img_shape` 是图像的维度（如 `(channels, height, width)`），`np.prod(img_shape)` 计算图像的总像素数，将图像扁平化为一维向量。
- `nn.LeakyReLU(0.2, inplace=True)`：Leaky ReLU 激活函数，带有 0.2 的负斜率，避免“死神经元”问题。
- `nn.Linear(512, 256)`：全连接层，将 512 维度的输入映射到 256 维。
- `nn.LeakyReLU(0.2, inplace=True)`：再次使用 Leaky ReLU 激活函数。
- `nn.Linear(256, 1)`：最后一层，将 256 维输入映射到 1 维输出，用来表示图像的真实性（真假）。
- `nn.Sigmoid()`：使用 Sigmoid 激活函数，将输出值限制在 [0, 1] 的范围，表示输入图像为“真实”的概率。输出接近 1 时表示更真实，接近 0 时表示更假。

### 3. `def forward(self, img)`
这是前向传播函数，定义了输入数据如何通过网络进行传播和处理：
- `img` 是输入的图像数据。
- `img_flat = img.view(img.size(0), -1)`：将输入图像 `img` 展平为一维向量，`img.size(0)` 是批量大小，`-1` 表示根据剩余维度自动计算展平后的大小。
- `validity = self.model(img_flat)`：展平后的图像数据通过判别器模型 `self.model`，输出一个 `validity` 值，表示输入图像的真假概率。
- `return validity`：返回计算的概率值 `validity`，作为判别器对输入图像的评估结果。

### 总结
这个判别器的作用是接收一个图像，将其展平为一维向量，经过几层全连接层和 LeakyReLU 激活，最终输出一个介于 0 和 1 之间的值，表示图像是否为真实数据。通过 Sigmoid 函数将结果归一化为概率值，用于 GAN 的训练过程中对抗生成器。

判别器在 GAN 中的任务是最大化区分生成器生成的假图像和真实数据，它通过学习能够更好地区分真假样本，从而提升生成器生成数据的质量。

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(int(np.prod(img_shape)), 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)

        return validity
